# <img style="float: left; padding-right: 20px; height: 70px" src="https://i.imgur.com/cMzxwTN.jpg"> Fundamentos de Machine Learning
## Actividad III, Sesión 2

<h4>

**Universidad Privada Boliviana**<br/>
**Diplomado en Machine Learning y Ciencia de Datos**<br/>

<h4>

<div style="line-height: 1.5">

### ■ Árboles Multivariables y Curvas de Complejidad

En esta actividad, entrenaremos modelos con **todas las variables predictoras** y analizaremos el fenómeno del sobreajuste mediante tablas y gráficas iterativas.

#### Celda 1: Integración de Todas las Variables Predictoras
*Retomaremos las matrices completas (`X_train_full` y `X_test_full`) que preparamos*



In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar los datos desde los archivos CSV
train_df = pd.read_csv('county_election_train.csv')
test_df = pd.read_csv('county_election_test.csv')

print(f"Dimensión de Entrenamiento: {train_df.shape}")
print(f"Dimensión de Prueba: {test_df.shape}")


# Crear la variable objetivo 'winner' (1 = Trump, 0 = Clinton)
train_df['winner'] = (train_df['trump'] > train_df['clinton']).astype(int)
test_df['winner'] = (test_df['trump'] > test_df['clinton']).astype(int)


In [ ]:
# Asignar los vectores de etiquetas (Target)
y_train = train_df['winner']
y_test = test_df['winner']

# Definir las columnas que NO son características predictoras
cols_to_drop = ['state', 'county', 'fipscode', 'trump', 'clinton', 'votergap', 'winner']

# Generar la matriz X eliminando las columnas no deseadas
X_train_full = train_df.drop(columns=cols_to_drop)
X_test_full = test_df.drop(columns=cols_to_drop)


In [ ]:
X_train_full.head()

In [ ]:

# Asignar las matrices completas para esta sección
X_train_all = ____
X_test_all = ____

# Verificar la dimensionalidad (cantidad de variables)
print(f"Cantidad de variables a utilizar: {X_train_all.____[1]}")


<div style="line-height: 1.5">

#### Celda 2: Entrenamiento Múltiple (D=2, 10, 15)
Instanciaremos tres clasificadores distintos aumentando progresivamente la profundidad máxima permitida. Esto nos ayudará a observar matemáticamente el deterioro de la generalización.


In [ ]:

# Instanciar modelos con 3 profundidades distintas
clf_d2 = DecisionTreeClassifier(max_depth=____, random_state=42)
clf_d10 = DecisionTreeClassifier(max_depth=____, random_state=42)
clf_d15 = DecisionTreeClassifier(max_depth=____, random_state=42)

# Entrenar los modelos con la matriz de datos completa
clf_d2.____(X_train_all, y_train)
clf_d10.____(X_train_all, y_train)
clf_d15.____(X_train_all, y_train)


<div style="line-height: 1.5">

#### Celda 3: Análisis de Rendimiento (PrettyTable y Curvas de Aprendizaje)


In [ ]:

from prettytable import ____

# 1. Crear y estructurar la tabla
tabla = PrettyTable()
tabla.field_names = ["Profundidad", "Train Score", "Test Score"]

tabla.____(["D=2", round(clf_d2.score(X_train_all, y_train), 4), round(clf_d2.score(X_test_all, y_test), 4)])
tabla.____(["D=10", round(clf_d10.score(X_train_all, y_train), 4), round(clf_d10.score(X_test_all, y_test), 4)])
tabla.____(["D=15", round(clf_d15.score(X_train_all, y_train), 4), round(clf_d15.score(X_test_all, y_test), 4)])
print(tabla)

# ==========================================
# 2. Bucle para generar la Gráfica de Complejidad
# ==========================================
train_scores = []
test_scores = []
profundidades = range(1, 21)

for d in profundidades:
    modelo = DecisionTreeClassifier(max_depth=____, random_state=42)
    modelo.fit(X_train_all, y_train)
    train_scores.____(modelo.score(X_train_all, y_train))
    test_scores.____(modelo.score(X_test_all, y_test))

# Graficar los resultados
plt.figure(figsize=(10, 6))
plt.____(profundidades, train_scores, label='Train Accuracy', marker='o')
plt.____(profundidades, test_scores, label='Test Accuracy', marker='s')
plt.title("Curva de Aprendizaje: Overfitting vs Profundidad")
plt.xlabel("Profundidad Máxima (max_depth)")
plt.ylabel("Precisión (Accuracy)")
plt.xticks(profundidades)
plt.legend()
plt.grid(True)
plt.show()


<div style="line-height: 1.5">

### ■ Análisis Avanzado: Feature Importance y Poda (Pruning)

#### Celda 1: Extracción de la Importancia de Características
Durante el entrenamiento, el árbol registra internamente cuánta *Ganancia de Información* aporta cada variable cada vez que es utilizada para dividir un nodo. 

Al finalizar, normaliza estos valores para que sumen 1.0 (o 100%). Esto nos permite graficar un ranking objetivo de las variables predictoras.


In [ ]:

# Entrenamos un modelo profundo base para evaluar la importancia
modelo_completo = DecisionTreeClassifier(random_state=42)
modelo_completo.fit(X_train_all, y_train)

# 1. Extraer los valores matemáticos de importancia
importancias = modelo_completo.____

# 2. Crear un DataFrame uniendo nombres de columnas con sus importancias
df_imp = pd.DataFrame({
    'Caracteristica': X_train_all.____,
    'Importancia': importancias
})

# 3. Ordenar de mayor a menor
df_imp = df_imp.sort_values(by='Importancia', ascending=False)

# 4. Generar el gráfico de barras horizontales
plt.figure(figsize=(10, 6))
sns.____(x='Importancia', y='Caracteristica', data=df_imp, palette='viridis')
plt.title('Importancia de Variables (Feature Importance)')
plt.xlabel('Fracción de Reducción de Impureza de Gini')
plt.ylabel('Variables Sociodemográficas')
plt.show()


<div style="line-height: 1.5">

#### Celda 2: Teoría Intuitiva de `ccp_alpha` (Poda por Costo-Complejidad)



**El Parámetro `ccp_alpha`:**
Este es un parámetro de penalización matemática (similar a la regularización C en regresión logística).
* Si `ccp_alpha = 0.0`: No hay penalización. El árbol mantiene todas sus ramas (Máxima complejidad = Riesgo de Overfitting).
* Si aumentamos `ccp_alpha` (ej. 0.01, 0.05): El algoritmo evaluará si una rama específica realmente ayuda a clasificar o si es solo "ruido". Si la rama aporta muy poca ganancia de información frente a la penalización, la rama se corta y se convierte en una hoja plana.




#### Celda 3: Implementación en Código de la Poda (Pruning)
Vamos a entrenar un nuevo árbol de decisión aplicando el parámetro `ccp_alpha` y compararemos su profundidad y rendimiento con el modelo completo original.


In [ ]:

# Instanciar el modelo con el hiperparámetro de poda de costo-complejidad
arbol_podado = DecisionTreeClassifier(random_state=42, ____=0.015)

# Ajustar el modelo podado a los datos
arbol_podado.fit(X_train_all, y_train)

# Imprimir estadísticas de complejidad
print("Profundidad Árbol Original (Sin podar):", modelo_completo.____())
print("Profundidad Árbol Podado:", arbol_podado.____())
print("Hojas Árbol Podado:", arbol_podado.get_n_leaves())
print("-" * 30)

# Comparar Accuracy (Generalización) en datos de Test
print("Test Score Original:", modelo_completo.score(X_test_all, y_test).round(4))
print("Test Score Podado:", arbol_podado.score(X_test_all, y_test).round(4))


<div style="line-height: 1.5">

**Diagnóstico Esperado:** Al ejecutar el código, notará que el `arbol_podado` reduce drásticamente su profundidad (de quizás 20+ a solo 3 o 4 niveles), y como resultado, el `Test Score` mejora significativamente respecto al modelo complejo.



<div style="line-height: 1.5">

#### Celda 4: Otros Criterios Topológicos de Parada
Además de `max_depth` y la poda avanzada (`ccp_alpha`), la API de `scikit-learn` ofrece:

* **`min_samples_leaf`:** Exige que cualquier nodo final (hoja) contenga un número mínimo de muestras. 
* **`max_leaf_nodes`:** El árbol  se detendrá  una vez que alcance la cantidad máxima de hojas indicadas.



In [ ]:

# Instanciar árbol obligando a tener al menos 30 condados por hoja 
# y permitiendo un máximo absoluto de 15 hojas finales.
arbol_restringido = DecisionTreeClassifier(
    random_state=42, 
    ____=30, 
    ____=15
)

# Ajuste y evaluación
arbol_restringido.fit(X_train_all, y_train)

print("Profundidad alcanzada:", arbol_restringido.get_depth())
print("Número de hojas reales:", arbol_restringido.get_n_leaves())
print("Score Test (Restringido):", arbol_restringido.score(X_test_all, y_test).round(4))

